# 02 · IIJP vs IPC: cuánto difieren y por qué

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/IPC_jubilados/blob/main/notebooks/02_iijp_vs_ipc.ipynb)

**Método.** Índice de canasta fija (Lowe), el mismo concepto que el IPC: las cantidades de la canasta
de 2017/18 se valorizan cada mes con los índices de precios por división del INDEC. La ponderación
efectiva de cada rubro se mueve con sus precios relativos.

El TP original aplicaba siempre las mismas ponderaciones a las variaciones mensuales. Eso no es una
canasta fija: sobrepondera los rubros que venían atrasados y exagera la brecha cuando hay ajustes de
precios relativos grandes (2024).

In [ ]:
# Configuracion: en Colab clona (o actualiza) el repo; en local usa la carpeta del repo
import os, sys, subprocess
REPO = 'https://github.com/santiagoriverti/IPC_jubilados.git'
if 'google.colab' in sys.modules:
    RAIZ = '/content/IPC_jubilados'
    if os.path.exists(RAIZ):
        subprocess.run(['git', '-C', RAIZ, 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RAIZ], check=True)
else:  # subir desde la carpeta actual hasta encontrar el repo
    RAIZ = os.getcwd()
    while not os.path.exists(os.path.join(RAIZ, 'src', 'pipeline.py')) and os.path.dirname(RAIZ) != RAIZ:
        RAIZ = os.path.dirname(RAIZ)
sys.path.insert(0, RAIZ)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src import pipeline, graficos, indices, ponderaciones as pond
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 30)

In [ ]:
N_BOOTSTRAP = 300
R = pipeline.calcular(n_bootstrap=N_BOOTSTRAP)
niv, vm = R['niveles'], R['var_mensual']

## Validación: replicar el IPC oficial

Con las ponderaciones del INDEC y el mismo cálculo se reproduce el nivel general nacional con un error
máximo de ~0,16% en casi 10 años. Si el método no replicara el IPC, no serviría para compararlo.

In [ ]:
v = R['validacion']
print(f"Error maximo: {v['error_pct'].abs().max():.3f}% | ultimo mes: {v['error_pct'].iloc[-1]:.3f}%")
display(v.tail(6))

## Variación mensual (%)

In [ ]:
display(vm[['IPC oficial', 'IPC ENGHo 17/18', 'IIJP', 'IIJP jub. bajos', 'IIJP método TP']].tail(13))

## La brecha del TP era en buena parte un artefacto del método

Desde noviembre de 2023 la planilla del TP acumula una brecha de 3,8% (con un error: en febrero 2024
falta la división Vivienda). El mismo método sin ese error da 6,3%; con canasta fija, la brecha es
**2,4%** a mayo 2026, y la banda de incertidumbre muestral no incluye el valor del TP.

In [ ]:
fig = graficos.g02_brecha_nov23(R); plt.show()

## Brecha por período y sus dos componentes

La brecha IIJP vs IPC oficial combina dos cosas distintas:

1. **Efecto edad**: IIJP vs un IPC recalculado con la canasta de *todos* los hogares de la ENGHo 2017/18.
2. **Efecto canasta vieja**: ese IPC 2017/18 vs el IPC oficial (ENGHo 2004/05).

In [ ]:
fig = graficos.g03_brecha_historica(R); plt.show()

In [ ]:
p = R['periodos']
cols = ['periodo', 'desde', 'hasta', 'IPC oficial (%)', 'IIJP (%)', 'brecha IIJP vs IPC oficial (%)',
        'brecha IPC ENGHo 17/18 vs IPC oficial (%)', 'brecha IIJP método TP vs IPC oficial (%)']
display(p[cols])

In [ ]:
if 'bootstrap_brechas' in R:
    display(Markdown('**Intervalos de 95% por bootstrap (incertidumbre muestral de la ENGHo):**'))
    display(R['bootstrap_brechas'])

**Lectura.** La brecha cambia de signo según el período: positiva con Macri (+2,8%), negativa con
Fernández (−4,0%, tarifas congeladas) y positiva desde fines de 2023 (+2,7%). Desde noviembre de 2023
**el efecto edad es negativo** (la canasta de los jubilados subió algo *menos* que la de todos los
hogares): lo que empuja la brecha es que el IPC oficial usa una canasta de 2004/05 que subpondera
vivienda y servicios y sobrepondera ropa. Una corrección permanente del 3,8% (propuesta del TP) no
tiene sustento en los datos.

## ¿La diferencia es sistemática? (test de la media mensual)

Diferencia mensual en logaritmos entre los índices, con intervalo de 95% robusto a autocorrelación
(Newey-West). Si el intervalo incluye el 0, no hay evidencia de una brecha sistemática en ese período.

In [ ]:
display(R['brecha_estadistica'])

## Qué divisiones explican la brecha

Aporte de cada división: (ponderación efectiva IIJP − ponderación efectiva de la otra canasta) ×
(variación de la división − variación promedio). Los aportes suman la brecha.

In [ ]:
fig = graficos.g04_descomposicion(R, 'nov23_ultimo'); plt.show()
display(R['descomposicion']['nov23_ultimo'])

In [ ]:
d = R['descomposicion']['dic16_ultimo']
print({k: round(v, 2) if isinstance(v, float) else v for k, v in d.attrs.items()})
display(d)